In [16]:
import kagglehub

path = kagglehub.competition_download('titanic', output_dir="./datasets/titanic")

print("Path to competition files:", path)

100%|████████████████████████████████████████████████████████████████████████| 34.1k/34.1k [00:00<00:00, 393kB/s]

Extracting files...
Path to competition files: ./datasets/titanic


In [86]:
import pandas as pd
from pathlib import Path

path = './datasets/titanic'
dataset_path = Path(path)

train_df = pd.read_csv(dataset_path / "train.csv")
test_df = pd.read_csv(dataset_path / "test.csv")
sub = pd.read_csv(dataset_path / "gender_submission.csv")

In [87]:
# train_df

In [88]:
train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


In [89]:
train_df.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [90]:
train_df.columns

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='str')

In [91]:
train_df.shape

(891, 12)

In [92]:
train_df.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [93]:
train_df.duplicated().sum()

np.int64(0)

In [94]:
train_df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


In [95]:
train_df_passid = train_df['PassengerId']
train_df = train_df.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'])

In [96]:
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder

ord_encoder = OrdinalEncoder(categories=[['male', 'female']])

train_df[['Sex']] = ord_encoder.fit_transform(train_df[['Sex']])

In [97]:
age_mean = train_df['Age'].median()
train_df['Age'] = train_df['Age'].fillna(age_mean)

In [98]:
embarked = train_df["Embarked"].mode()[0]

train_df["Embarked"] = train_df["Embarked"].fillna(embarked)

In [99]:
train_df

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,0.0,22.0,1,0,7.2500,S
1,1,1,1.0,38.0,1,0,71.2833,C
2,1,3,1.0,26.0,0,0,7.9250,S
3,1,1,1.0,35.0,1,0,53.1000,S
4,0,3,0.0,35.0,0,0,8.0500,S
...,...,...,...,...,...,...,...,...
886,0,2,0.0,27.0,0,0,13.0000,S
887,1,1,1.0,19.0,0,0,30.0000,S
888,0,3,1.0,28.0,1,2,23.4500,S
889,1,1,0.0,26.0,0,0,30.0000,C


In [100]:
one_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

enc_val = one_encoder.fit_transform(train_df[['Embarked']])
new_col = one_encoder.get_feature_names_out(['Embarked'])

In [102]:
df_enc = pd.DataFrame(enc_val, columns=new_col, index=train_df.index)

train_df = pd.concat([train_df.drop(columns=['Embarked']), df_enc], axis=1)

train_df

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked_C,Embarked_Q,Embarked_S
0,0,3,0.0,22.0,1,0,7.2500,0.0,0.0,1.0
1,1,1,1.0,38.0,1,0,71.2833,1.0,0.0,0.0
2,1,3,1.0,26.0,0,0,7.9250,0.0,0.0,1.0
3,1,1,1.0,35.0,1,0,53.1000,0.0,0.0,1.0
4,0,3,0.0,35.0,0,0,8.0500,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...
886,0,2,0.0,27.0,0,0,13.0000,0.0,0.0,1.0
887,1,1,1.0,19.0,0,0,30.0000,0.0,0.0,1.0
888,0,3,1.0,28.0,1,2,23.4500,0.0,0.0,1.0
889,1,1,0.0,26.0,0,0,30.0000,1.0,0.0,0.0


In [103]:
X_train = train_df.drop(columns=['Survived'])
y_train = train_df['Survived']

In [104]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)

In [108]:
X_train

array([[ 0.82737724, -0.73769513, -0.56573646, ..., -0.48204268,
        -0.30756234,  0.61583843],
       [-1.56610693,  1.35557354,  0.66386103, ...,  2.0745051 ,
        -0.30756234, -1.62380254],
       [ 0.82737724,  1.35557354, -0.25833709, ..., -0.48204268,
        -0.30756234,  0.61583843],
       ...,
       [ 0.82737724,  1.35557354, -0.1046374 , ..., -0.48204268,
        -0.30756234,  0.61583843],
       [-1.56610693, -0.73769513, -0.25833709, ...,  2.0745051 ,
        -0.30756234, -1.62380254],
       [ 0.82737724, -0.73769513,  0.20276197, ..., -0.48204268,
         3.25137334, -1.62380254]], shape=(891, 9))

In [109]:
from sklearn.linear_model import LogisticRegression

LR = LogisticRegression()

LR.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following

In [112]:
feature_columns = train_df.drop(columns=["Survived"]).columns.tolist()

fare_median = train_df["Fare"].median()


def prepare_test_data(test_df):
    df = test_df.copy()

    passenger_ids = df["PassengerId"].copy()

    df = df.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])

    df["Age"] = df["Age"].fillna(age_mean)
    df["Embarked"] = df["Embarked"].fillna(embarked)
    df["Fare"] = df["Fare"].fillna(fare_median)

    df[["Sex"]] = ord_encoder.transform(df[["Sex"]])

    enc_val = one_encoder.transform(df[["Embarked"]])

    enc_df = pd.DataFrame(
        enc_val,
        columns=one_encoder.get_feature_names_out(["Embarked"]),
        index=df.index
    )

    df = pd.concat(
        [df.drop(columns=["Embarked"]), enc_df],
        axis=1
    )
    
    df = df.reindex(columns=feature_columns, fill_value=0)

    return passenger_ids, df

In [113]:
test_ids, X_test = prepare_test_data(test_df)

X_test_scaled = scaler.transform(X_test)
predictions = LR.predict(X_test_scaled)

In [114]:
predictions

array([0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0,
       1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1,
       1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1,
       1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1,
       1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0,
       0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0,
       1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0,
       0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1,
       1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1,
       0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0,
       1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1,
       0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1,
       0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0,
       0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0,

In [116]:
submission = pd.DataFrame({
    "PassengerId": test_ids,
    "Survived": predictions
})

submission.to_csv("./datasets/titanic/submission.csv", index=False)

In [117]:
print(submission.head())
print(submission.shape)
print(submission.columns.tolist())
print(submission.isnull().sum())
print(submission["Survived"].value_counts())

   PassengerId  Survived
0          892         0
1          893         0
2          894         0
3          895         0
4          896         1
(418, 2)
['PassengerId', 'Survived']
PassengerId    0
Survived       0
dtype: int64
Survived
0    262
1    156
Name: count, dtype: int64
